In [3]:
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sys.path.append("/home/dante/Documents/opendc/graph-greenifier-github")
sys.path.append("../")


from plottingTools import utils

# 1. Topology

- Topologies define the available hardware of a datacenter

- Defined using JSON format

- In this demo we compare two topologies
    - [Small](topologies/small_topology.json)
    - [Large](topologies/large_topology.json)
    - [Greenifier](topologies/greenifier_topology.json)

# 2. Workloads

- In Graph Massivizer we use a different workload format that is similar to the output generated by the Graph Greenifier: [Greenifier Workload](workloadTraces/greenifier/workload.json)

- A single JSON file defines all tasks, their requirements, and dependencies

# 3. Carbon Intensity

- Carbon Intensity: the amount of carbon emitted per unit of energy

- Graph Greenifier needs to know the carbon intensity

- A trace is provided with the carbon intensity during the workload

- Information gathered from ENTSO-E

In [26]:
df_carbon = pd.read_parquet("carbonTraces/carbon_2022.parquet")
df_carbon.head()

,timestamp,carbon_intensity
0,2021-12-31 23:00:00,168.138693
1,2021-12-31 23:15:00,167.050014
2,2021-12-31 23:30:00,164.552936
3,2021-12-31 23:45:00,167.493769
4,2022-01-01 00:00:00,164.517793


# 4. Scenario

- A scenario defines what the Graph Greenifier should run, and how.

- Scenarios are defined using a JSON format

- [greenifier scenario](scenarios/greenifier_scenario.json)

- Parameters are defined as lists, this makes it easy to run multiple experiments

# 5. Running Greenifier

- Graph Greenifier is a terminal tool which can run directly. 
- Only a scenario is needed 

In [52]:
pathToScenario = "scenarios/greenifier_scenario.json"
subprocess.run(["../bin/Greenifier", "--scenario-path", pathToScenario])



 Running scenario: 0 


Simulating...   0% [                                       ] 0/1 (0:00:00 / ?) 



 Running scenario: 1 


Simulating... 100% [=================================] 1/1 (0:00:01 / 0:00:00) 
Simulating... 100% [=================================] 1/1 (0:00:00 / 0:00:00) 


CompletedProcess(args=['../bin/Greenifier', '--scenario-path', 'scenarios/greenifier_scenario.json'], returncode=0)

## 6. Output

In [32]:
pathToOutput = "output/greenifier"

df_host = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/host.parquet")
df_server = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/server.parquet")
df_service = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/service.parquet")

## 7. Aggregated results

- To properly compare the different experiments, we would like to aggregate them into meaningfull values.

### Performance

- For many operators, it is important to know what kind of performance to expect

- Graph Greenifier can provide many performance metrics:
    - Total runtime
    - Utilization

In [67]:
runtime = utils.getTotalRuntime(df_service) 
utilization = utils.getMeanUtilization(df_host)


print(f"The total runtime of the workload was {runtime}")
print(f"On average, the utilization of each host is {utilization * 100:.2f}%")

The total runtime of the workload was 0 days 17:56:12.251000
On average, the utilization of each host is 3.14%


### Sustianability

- Graph Greenifier can provide an insight into sustainabilily
- Lets calculate the total energy usage and carbon emissions 

In [68]:
energy_usage = utils.getTotalEnergyUsage(df_host, "kWh")
carbon_emissions = (df_host["carbon_emission"].sum() / 1000).round(2)

print(f"The data center used {energy_usage:.2f} kWh while running the workload")
print(f"The data center emitted {carbon_emissions:.2f} kg of carbon during the workload")

The data center used 12.19 kWh while running the workload
The data center emitted 2.89 kg of carbon during the workload


# 10. Graph Massivizer output

- For Graph Massivizer we need to provide choreographer with information

- Information is provided in JSON format

In [ ]:
import utils

output = utils.get_output(df_mem_host, df_mem_service, df_mem_server, save=True, exportName="output/greenifier_output.json")